# 09 — Anchor Post Characterisation (BART NLI)

Characterises the BART-selected anchor posts across all three subreddits.
BART inference now runs inside NB03 as part of anchor selection — this notebook
reads `anchor_posts.parquet` (which already contains BART fields) and produces
summary statistics and figures.

**Inputs:**
- `data/processed/{subreddit}/anchor_posts.parquet` for each subreddit in SUBREDDITS

**Outputs:**
- `figures/fig_nli_validation_scores.png` — BART score and label distributions

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                        NB09 CONFIG                          ║
# ╚══════════════════════════════════════════════════════════════╝

SUBREDDITS = ['gradadmissions', 'mscs', 'mba']

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
DATA    = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print('Repo root:', ROOT)

In [ ]:
# ── Load anchor posts from all subreddits (BART fields already present) ────
frames = []
for sub in SUBREDDITS:
    path = DATA / sub / 'anchor_posts.parquet'
    df = pd.read_parquet(path)
    df['subreddit'] = sub
    frames.append(df)
    print(f'r/{sub}: {len(df):,} anchor posts')

anchors = pd.concat(frames, ignore_index=True)
anchors['cycle'] = anchors['cycle'].astype(int)
anchors['bart_is_negative'] = anchors['bart_is_negative'].astype(bool)

print(f'\nTotal anchor posts: {len(anchors):,}')
print(anchors.groupby('subreddit')[['mean_mh_score', 'bart_top_score']].agg(['count', 'mean']).round(3))

In [ ]:
# ── (1) Summary statistics ─────────────────────────────────────────────────

total = len(anchors)

print('=' * 60)
print('ANCHOR POST SUMMARY (BART-selected)')
print('=' * 60)
print(f'Total anchor posts: {total:,}')

print()
print('--- Per-subreddit counts ---')
for sub, grp in anchors.groupby('subreddit'):
    print(f'  r/{sub:<18s}: {len(grp):,} posts')

print()
print('--- Top-label distribution (all subreddits) ---')
print(anchors['bart_top_label'].value_counts().to_string())

print()
print('--- Per-subreddit × top-label breakdown ---')
cross = anchors.groupby(['subreddit', 'bart_top_label']).size().unstack(fill_value=0)
print(cross.to_string())

print()
print('--- Anchor counts per cycle ---')
print(anchors.groupby(['subreddit', 'cycle']).size().unstack(fill_value=0).to_string())

print()
print('--- Mean BART top-negative-label score by subreddit ---')
print(anchors.groupby('subreddit')['bart_top_neg_score']
      .agg(['mean', 'median', 'std']).round(3).to_string())

print()
print('--- Mean SVM mh_score by subreddit ---')
print(anchors.groupby('subreddit')['mean_mh_score']
      .agg(['mean', 'median', 'std']).round(3).to_string())

In [ ]:
# ── (2) Score distribution plots ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram of top-negative-label score (BART confidence) per subreddit
ax = axes[0]
for sub, grp in anchors.groupby('subreddit'):
    ax.hist(grp['bart_top_neg_score'], bins=25, alpha=0.55, label=f'r/{sub}',
            edgecolor='white', linewidth=0.4)
ax.axvline(0.5, color='crimson', linestyle='--', linewidth=1.3, label='p = 0.5')
ax.set_xlabel('BART top-negative-label entailment score', fontsize=11)
ax.set_ylabel('Anchor posts', fontsize=11)
ax.set_title('Distribution of BART confidence scores\n(anchor posts, all subreddits)', fontsize=11)
ax.legend(fontsize=9)

# Right: SVM mean_mh_score vs BART top_neg_score, coloured by label
label_colours = {
    'negative admissions outcome':    '#E91E63',
    'rejection or funding loss':      '#9C27B0',
    'giving up on graduate school':   '#FF5722',
}
default_colour = '#9E9E9E'
colours = anchors['bart_top_label'].map(label_colours).fillna(default_colour)
ax = axes[1]
ax.scatter(anchors['mean_mh_score'], anchors['bart_top_neg_score'],
           c=colours, alpha=0.5, s=16, linewidths=0)
ax.set_xlabel('SVM mean_mh_score', fontsize=11)
ax.set_ylabel('BART top-negative-label score', fontsize=11)
ax.set_title('SVM score vs BART confidence\n(coloured by BART label)', fontsize=11)
for label, colour in label_colours.items():
    ax.scatter([], [], c=colour, label=label, s=30)
ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
out_path = FIG_DIR / 'fig_nli_validation_scores.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path.name}')

In [ ]:
# ── (3) High-confidence examples by BART label ────────────────────────────

pd.set_option('display.max_colwidth', 500)

PRINT_CHARS = 600
N_EXAMPLES  = 5

def _print_examples(df, section_title):
    print('=' * 72)
    print(section_title)
    print('=' * 72)
    for _, row in df.iterrows():
        hdr = (f'[r/{row.subreddit} | cycle={row.cycle} | '
               f'SVM={row.mean_mh_score:.3f} | '
               f'BART: {row.bart_top_label} ({row.bart_top_score:.2f})]')
        print()
        print(hdr)
        print(str(row.clean_text)[:PRINT_CHARS])
        print('-' * 60)

for label in sorted(anchors['bart_top_label'].unique()):
    subset = (
        anchors[anchors['bart_top_label'] == label]
        .nlargest(N_EXAMPLES, 'bart_top_score')
        .reset_index(drop=True)
    )
    _print_examples(subset, f'Top {N_EXAMPLES} by BART confidence: "{label}"')
    print()

In [ ]:
# ── (4) Per-cycle anchor counts and score summary ─────────────────────────

print('--- Anchor posts per cycle per subreddit ---')
cycle_table = anchors.groupby(['subreddit', 'cycle']).size().unstack(fill_value=0)
print(cycle_table.to_string())

print()
print('--- BART top-neg score by subreddit × label ---')
score_table = (
    anchors.groupby(['subreddit', 'bart_top_label'])['bart_top_neg_score']
    .agg(['count', 'mean', 'std'])
    .round(3)
)
print(score_table.to_string())

print()
print('--- SVM scores of anchor posts (all subreddits) ---')
print(anchors[['anx_score', 'dep_score', 'str_score', 'mean_mh_score']]
      .describe().round(3).to_string())